#### Reads workdone from google sheets file tracker and sends progress emails to supervisor

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict,Any
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [ ]:
from typing import List, Dict, Any
from google.oauth2 import service_account
from openai import AsyncOpenAI
from googleapiclient.discovery import build

In [ ]:
load_dotenv(override=True)

In [ ]:
summary_instructions = """ You are a summarisation agent. Your job has TWO steps and you must complete BOTH:
STEP 1: Generate a SHORT plain text summary titled 'Summary:' of the tasks provided.
STEP 2: You MUST call the handoff tool to transfer to the Email Agent. Do not just say you are handing off — actually use the handoff tool. This is mandatory."""

email_instructions = """ You are an email agent, you receive a professional task summary. Your job has TWO SEQUENTIAL STEPS, and you must do them IN ORDER:
STEP 1: Use the format_email tool to format the email
STEP 2: Use the send_email tool to send the email """

manager_instructions = "Fetch the latest 5 tasks using the tool. HANDOFF the result to the Summarisation Agent."

In [ ]:
import os
print(os.getcwd())
print(os.path.abspath("service_account.json"))

In [ ]:
@function_tool
def read_tasks_from_google_sheet() -> Dict[str, Any]:
    """Reads Google Sheet and returns a formatted text summary of the last 5 task entries."""

    try:
        SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
        SERVICE_ACCOUNT_FILE = r"C:\Users\92310\Desktop\Projects\agents\2_openai\service_account.json"
        SPREADSHEET_ID = "1UFvac9xxRWI-lNEhy9TLLNEU_IaUYstGPV6YiT-AxhQ"  # hardcoded

        creds = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
        service = build('sheets', 'v4', credentials=creds)
        sheet = service.spreadsheets()

        result = sheet.values().get(
            spreadsheetId=SPREADSHEET_ID,
            range="Tasks!B3:H"  # hardcoded with sheet name
        ).execute()

        values = result.get("values", [])

        if len(values) < 4:
            return {"status": "success", "summary": "No task entries found."}

        headers = values[0]
        rows = values[3:]

        rows = [r for r in rows if any(cell.strip() for cell in r if cell)]

        if not rows:
            return {"status": "success", "summary": "No task entries found."}

        last_5 = rows[-5:]

        formatted_entries = []
        for row in last_5:
            row_dict = dict(zip(headers, row))
            entry_text = (
                f"Type: {row_dict.get('Type', 'N/A')}\n"
                f"Task: {row_dict.get('Task', 'N/A')}\n"
                f"User: {row_dict.get('User', 'N/A')}\n"
                f"Status: {row_dict.get('Status', 'N/A')}\n"
                f"Note: {row_dict.get('Note', 'N/A')}\n"
                f"Start Date: {row_dict.get('Start date', 'N/A')}\n"
                f"Due On: {row_dict.get('Due on', 'N/A')}\n"
                f"{'-'*40}"
            )
            formatted_entries.append(entry_text)

        return {
            "status": "success",
            "entries_returned": len(last_5),
            "text": "\n".join(formatted_entries)
        }

    except Exception as e:
        return {"status": "error", "message": str(e)}

In [ ]:
@function_tool
def format_email(summary_text: str) -> str:
    """ Formats a task summary into a simple email body. """

    email_body = (
        "Hello Team,\n\n"
        "Please find below the latest work tracker summary:\n\n"
        f"{summary_text}\n\n"
        "Best regards,\n"
        "Work Tracker Automation"
        )

    return email_body

In [ ]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("zoya5a.bclr@gmail.com")  
    to_email = To("zoyahammadk@gmail.com")  
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Task Update Email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return "Email sent successfully"

In [ ]:
email_agent = Agent(name="Email Agent",instructions=email_instructions,tools=[format_email, send_email])

In [ ]:
summarisation_agent = Agent(name="Summarisation Agent", instructions=summary_instructions, model="gpt-4o-mini", handoffs=[email_agent])

In [ ]:
manager_agent = Agent(name="Task Manager",instructions=manager_instructions, tools=[read_tasks_from_google_sheet],handoffs=[summarisation_agent])

In [ ]:
with trace('Spreadsheet Updates'):
    result = await Runner.run(manager_agent, "Show me the last 5 tasks from the sheet")